# Lesson 5 · Training a classifier, and watching it fail

<a href="https://colab.research.google.com/github/Mary-Vadi/rag/blob/main/lessons/05_training_a_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**In this lesson you will**

- fine-tune **ModernBERT**, a modern transformer model, to detect machine-generated text
- test it on the kind of text it was trained on (RAID) and on a different kind (MAGE)
- meet **domain shift**, the main problem retrieval-augmented detection tries to soften

**Runtime:** a **GPU is needed**. In Colab choose Runtime → Change runtime type → T4 GPU. Training takes a few minutes.

## 1. What does fine-tuning mean?

Training a language model from scratch takes enormous amounts of text and computing power. Instead we start from a model that has already been **pre-trained**: it has read a huge amount of text and learned a lot about how language works.

**Fine-tuning** adapts that model to our task:

1. Put a small new layer on top of the model, a **classification head**, that turns the model's output into two scores: human and machine.
2. Show the model labelled examples. After each small batch, nudge all of its weights slightly so that it would have been a little more right.
3. Repeat for one or more passes over the data (**epochs**).

We use [`answerdotai/ModernBERT-base`](https://huggingface.co/answerdotai/ModernBERT-base), a 2024 update of the classic BERT model that is fast and handles long texts well. It is the same model as in the original experiment.

## 2. Setup

In [ ]:
# Setup: run this cell first.
# In Colab it downloads the course code and installs the extra libraries.
# On your own computer (from the lessons/ folder) it just makes the code importable.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/rag"):
        !git clone -q https://github.com/Mary-Vadi/rag.git /content/rag
        !pip install -q -r /content/rag/requirements.txt
    %cd /content/rag
elif os.path.basename(os.getcwd()) == "lessons":
    %cd ..
sys.path.insert(0, os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.model_selection import train_test_split

from ragdetect.classifier import predict_proba, train_classifier
from ragdetect.data import LABEL_NAMES, load_mage, load_raid
from ragdetect.evaluation import compare, evaluate, plot_confusion

print("GPU available:", torch.cuda.is_available())

If that says `False`, switch the runtime to a GPU now (Runtime → Change runtime type), then run the notebook again from the top. Training on a CPU would take a very long time.

## 3. The training data

We load 4,000 RAID texts (the same sample as lesson 1) and split them 80 / 10 / 10 into **train**, **validation** and **test** sets. `stratify` keeps the human/machine balance the same in all three.

In [ ]:
raid = load_raid(n=4000)
raid_train, raid_rest = train_test_split(raid, test_size=0.2, stratify=raid["label"], random_state=42)
raid_val, raid_test = train_test_split(raid_rest, test_size=0.5, stratify=raid_rest["label"], random_state=42)
print(f"train {len(raid_train)}, validation {len(raid_val)}, test {len(raid_test)}")

In [ ]:
raid_train["domain"].value_counts()

> **Every training text is a scientific abstract.** Remember this: it's the key to what happens later in this lesson.

## 4. Fine-tune

`train_classifier` in `ragdetect/classifier.py` is about 50 lines of standard Hugging Face code. Open it and read it alongside this table of the settings that matter most:

| Setting | Value | What it means |
|---|---|---|
| `epochs` | 1 | how many times the model sees each training text |
| `learning_rate` | 2e-5 | how big each nudge to the weights is; fine-tuning uses small values |
| `batch_size` | 16 | how many texts the model looks at before each nudge |
| `max_length` | 256 | tokens per text; longer texts are cut off |

After each epoch the model is scored on the validation set, and the best version is kept and saved to `models/classifier`.

In [ ]:
model_dir = train_classifier(raid_train, raid_val, output_dir="models/classifier", epochs=1, max_length=256)

You'll probably see a "LOAD REPORT" saying `classifier.weight` and `classifier.bias` are **MISSING**. That's expected: they are the brand-new classification head from step 1 above, which starts out random and is exactly what training fills in.

## 5. Test 1: more RAID abstracts (in-domain)

First, the fair test: texts the model has never seen, but of the same kind as its training data.

In [ ]:
raid_test_proba = predict_proba(model_dir, raid_test["text"])
results = {"classifier on RAID test (in-domain)": evaluate(raid_test["label"], raid_test_proba)}
compare(results)

## 6. Test 2: MAGE (out-of-domain)

Now the real-world test: MAGE's Reddit posts, news, stories, reviews and more, written by people and by 27 different models.

In [ ]:
mage_test = load_mage("test", n=2000)
mage_proba = predict_proba(model_dir, mage_test["text"])
results["classifier on MAGE test (out-of-domain)"] = evaluate(mage_test["label"], mage_proba)
compare(results)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_confusion(raid_test["label"], raid_test_proba, title="RAID test (in-domain)", ax=axes[0])
plot_confusion(mage_test["label"], mage_proba, title="MAGE test (out-of-domain)", ax=axes[1])
plt.tight_layout()
plt.show()

How big is the gap between the two? In the full-size original experiment (see the `experiments/` folder), the same kind of model scored **98.8%** on RAID but only **53.8%** on MAGE, barely better than guessing. Nearly 9 out of 10 human MAGE texts were flagged as AI.

## 7. Why? Domain shift

A model learns whatever separates its two classes *in its training data*. Ours has only ever seen scientific abstracts, so it has learned "how AI-written abstracts differ from human-written abstracts". Some of that may be general, but much of it is specific to abstracts: their vocabulary, their structure, their tone.

When the input changes (Reddit arguments, news stories, restaurant reviews), those clues stop working, and the model falls back on guesses. This is **domain shift**: the data a model is used on differs from the data it was trained on. High accuracy on a test set drawn from the training distribution tells you very little about what happens outside it.

Let's see *where* it fails, by grouping MAGE's test texts by their original dataset:

In [ ]:
mage_test = mage_test.assign(
    origin=mage_test["source"].str.split("_").str[0],
    correct=((mage_proba >= 0.5).astype(int) == mage_test["label"]),
)
mage_test.groupby("origin")["correct"].agg(["mean", "size"]).rename(columns={"mean": "accuracy", "size": "texts"}).sort_values("accuracy").round(3)

And how *confident* is the model? Here is the distribution of its `P(machine)` scores on MAGE, split by the true label:

In [ ]:
plt.figure(figsize=(7, 3.5))
for label, name in enumerate(LABEL_NAMES):
    plt.hist(mage_proba[mage_test["label"] == label], bins=20, range=(0, 1), alpha=0.6, label=f"really {name}")
plt.axvline(0.5, color="black", linestyle="--", linewidth=1)
plt.xlabel("classifier's P(machine)")
plt.ylabel("number of texts")
plt.title("MAGE test: classifier scores")
plt.legend()
plt.show()

If many human texts sit far to the right, near 1.0, the model isn't just wrong about them, it's **confidently** wrong. Keep this picture in mind: in lesson 6 it explains why the original experiment's retrieval step could barely help.

## 8. A note on the original experiment

The original experiment trained on the **first 10% of RAID's training file**. Because that file is sorted by domain, spot checks suggest those rows are almost entirely scientific abstracts too. So this lesson reproduces the same situation, just at a smaller scale. A fairer experiment would sample RAID texts from all of its domains, which requires downloading the whole 11.8 GB file.

## Check your understanding

<details>
<summary>1. What is domain shift?</summary>

When the data a model is used on differs from the data it was trained on: different topics, writing styles, text sources or generating models. Performance measured on the training distribution then overestimates performance in practice.
</details>

<details>
<summary>2. The classifier scores very highly on RAID. Why isn't that good news on its own?</summary>

Because the RAID test texts are the same kind of text as its training data. It shows the model learned its training data well, not that it learned what makes text machine-generated in general.
</details>

<details>
<summary>3. Name two ways to make the classifier more robust.</summary>

Train on more varied data (all of RAID's domains, or several datasets); include adversarial examples; or, as in this course, bring in knowledge about the new domain at prediction time through retrieval.
</details>

## Try it yourself

1. Train with `load_raid(n=4000, include_attacks=True)`. Does in-domain accuracy change? Does MAGE accuracy?
2. Train for 2 epochs. Does the RAID score go up? Does the MAGE score?

## Summary

- **Fine-tuning** adapts a pre-trained model to a new task with a small amount of labelled data.
- Our classifier is trained on RAID abstracts and tested on the much more varied MAGE.
- In-domain accuracy can be excellent while out-of-domain accuracy collapses: that's **domain shift**.
- A model can be confidently wrong on unfamiliar text.

**Next:** [Lesson 6 · Retrieval-augmented detection](06_retrieval_augmented_detection.ipynb), where we combine the classifier with retrieval.